In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTENC

# Load the raw dataset
df = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
print(f'Dataset shape: {df.shape}')


Dataset shape: (7043, 21)


In [2]:
df.head(2)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No


In [3]:
print(f"null col: {df.isnull().sum()}")
print(f"d dtypes : {df.dtypes}")

null col: customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64
d dtypes : customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object


In [4]:
# Convert TotalCharges from string to numeric, coercing blank strings to NaN
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce') 

# Check how many NaN values were created
print(f'Missing TotalCharges: {df["TotalCharges"].isna().sum()}')

# Fill NaN with the median value
df['TotalCharges']= df['TotalCharges'].fillna(df['TotalCharges'].median())
print(f'Missing after imputation: {df["TotalCharges"].isna().sum()}')

Missing TotalCharges: 11
Missing after imputation: 0


In [5]:
# Drop customerID since it has no predictive value
#df = df.drop(columns=['customerID'])
print(df[['tenure','TotalCharges']].head(2))
a = (df['tenure'] + 1 )
print(a[:4])
# Derived feature: average monthly charge (guards against division by zero)
df['AvgMonthlyCharge'] = df['TotalCharges'] / (df['tenure'] + 1)

# Derived feature: binary flag for premium support services
df['HasPremiumSupport'] = (
    (df['OnlineSecurity'] == 'Yes') | (df['TechSupport'] == 'Yes')
).astype(int)

print('New columns: AvgMonthlyCharge, HasPremiumSupport')
print(df[['AvgMonthlyCharge', 'HasPremiumSupport']].head())

   tenure  TotalCharges
0       1         29.85
1      34       1889.50
0     2
1    35
2     3
3    46
Name: tenure, dtype: int64
New columns: AvgMonthlyCharge, HasPremiumSupport
   AvgMonthlyCharge  HasPremiumSupport
0         14.925000                  0
1         53.985714                  1
2         36.050000                  1
3         40.016304                  1
4         50.550000                  0


In [6]:
# Encode the target column as 0/1
df['Churn'] = (df['Churn'] == 'Yes').astype(int)  # if churn is no then 0 , yes is 1

# One-hot encode all remaining object columns
cat_cols = df.select_dtypes(include='object').columns.tolist()
df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

print(f'Shape after encoding: {df.shape}')
print(f'Churn distribution:')
print(df['Churn'].value_counts())


Shape after encoding: (7043, 7075)
Churn distribution:
Churn
0    5174
1    1869
Name: count, dtype: int64


In [7]:
print(cat_cols)
df.head()

['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']


,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn,AvgMonthlyCharge,HasPremiumSupport,customerID_0003-MKNFE,customerID_0004-TLHLJ,customerID_0011-IGKFF,...,StreamingTV_No internet service,StreamingTV_Yes,StreamingMovies_No internet service,StreamingMovies_Yes,Contract_One year,Contract_Two year,PaperlessBilling_Yes,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,0,1,29.85,29.85,0,14.925000,0,False,False,False,...,False,False,False,False,False,False,True,False,True,False
1,0,34,56.95,1889.50,0,53.985714,1,False,False,False,...,False,False,False,False,True,False,False,False,False,True
2,0,2,53.85,108.15,1,36.050000,1,False,False,False,...,False,False,False,False,False,False,True,False,False,True
3,0,45,42.30,1840.75,0,40.016304,1,False,False,False,...,False,False,False,False,True,False,False,False,False,False
4,0,2,70.70,151.65,1,50.550000,0,False,False,False,...,False,False,False,False,False,False,True,False,True,False


In [8]:
import sys
sys.path.insert(0, '..')
from src.preprocessing import preprocess_data

# Reload the raw data and run the full pipeline
df_raw = pd.read_csv('../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv')
X_train_res, X_test, y_train_res, y_test, feature_names = preprocess_data(df_raw)

print(f'Training set (resampled): {X_train_res.shape}')
print(f'Test set: {X_test.shape}')
print('Training label distribution:')
print(y_train_res.value_counts())

Training set (resampled): (8278, 32)
Test set: (1409, 32)
Training label distribution:
Churn
0    4139
1    4139
Name: count, dtype: int64


In [9]:
# Save processed datasets for the modeling notebook
X_train_res.to_csv('../data/processed/X_train_resampled.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train_res.to_csv('../data/processed/y_train_resampled.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print('Saved to data/processed/')

Saved to data/processed/
